[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_3_5/17_tag_3_5_tabular_pipeline_preprocessing_tfrecords.ipynb)

# Tag 3-5 - 17 Komplette Pipeline: Imputation, Normalisierung, Encoding und TFRecords

In echten Projekten besteht Deep Learning nicht nur aus einem Modell. Eine robuste Pipeline braucht:

- sauberen Train/Validation/Test-Split,
- Imputation fehlender Werte,
- Normalisierung numerischer Features,
- Encoding kategorischer Features,
- speicherbares Preprocessing,
- effizientes Datenformat für Training.

Dieses Notebook zeigt eine vollständige kleine Tabular-Pipeline mit Keras-Preprocessing und TFRecords.


## Lesepfad für dieses Notebook

Dieses Notebook hat mehrere Ebenen. Die Reihenfolge ist wichtig:

1. `df`: Rohdaten als pandas DataFrame.
2. `train_df`, `val_df`, `test_df`: Split vor jeder Vorverarbeitung.
3. scikit-learn-Pipeline: leicht lesbare klassische Tabular-Pipeline.
4. `dataframe_to_dataset(...)`: wandelt pandas DataFrames in `tf.data.Dataset` um.
5. Keras-Preprocessing-Modell: dieselbe Vorverarbeitung direkt im Modell.
6. TFRecord-Teil: schreibt Trainingsdaten in eine Datei und liest sie wieder als Dataset.

Merksatz: Funktionen wie `dataframe_to_dataset`, `serialize_row` und `parse_example` werden erst definiert und danach direkt in den nächsten Zellen benutzt. Die Kommentare im Code markieren diese Übergänge.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


## Rohdaten mit fehlenden Werten

Wir simulieren einen Kundendatensatz. Das Ziel `churn` ist binär: Kundin oder Kunde kündigt ja/nein.

Wichtig: Fehlende Werte werden bewusst eingebaut, damit die Pipeline echte Vorverarbeitung enthält.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 3500

df = pd.DataFrame(
    {
        "age": rng.normal(43, 12, n).clip(18, 82),
        "monthly_spend": rng.gamma(shape=3.0, scale=22.0, size=n),
        "support_tickets": rng.poisson(1.2, n).astype(float),
        "contract": rng.choice(["monthly", "yearly", "two_year"], size=n, p=[0.55, 0.30, 0.15]),
        "region": rng.choice(["north", "south", "east", "west"], size=n),
        "payment": rng.choice(["card", "invoice", "paypal"], size=n, p=[0.55, 0.25, 0.20]),
    }
)

logit = (
    -1.2
    + 0.018 * (df["monthly_spend"] - 60)
    + 0.55 * (df["contract"] == "monthly").astype(float)
    + 0.35 * (df["support_tickets"] > 2).astype(float)
    - 0.45 * (df["payment"] == "card").astype(float)
    + rng.normal(0, 0.65, n)
)
proba = 1 / (1 + np.exp(-logit))
df["churn"] = rng.binomial(1, proba).astype("float32")

for col in ["age", "monthly_spend", "support_tickets", "contract", "payment"]:
    missing_mask = rng.random(n) < 0.07
    df.loc[missing_mask, col] = np.nan

df.head()


## Split vor der Vorverarbeitung

Alle Statistiken für Imputation, Normalisierung und Vokabulare dürfen nur auf Trainingsdaten gelernt werden. Sonst leakt Information aus Validation/Test in das Training.


In [ ]:
# Split 1: train_df wird komplett getrennt, bevor irgendeine Vorverarbeitung gelernt wird.
train_df, rest_df = train_test_split(
    df, test_size=0.30, random_state=RANDOM_STATE, stratify=df["churn"]
)
# Split 2: Der Rest wird halbiert in Validation und Test.
val_df, test_df = train_test_split(
    rest_df, test_size=0.50, random_state=RANDOM_STATE, stratify=rest_df["churn"]
)

# Diese Listen steuern ab jetzt alle Pipelines.
# Wenn ein Feature numerisch ist, landet es in numeric_features.
# Wenn es kategorisch ist, landet es in categorical_features.
numeric_features = ["age", "monthly_spend", "support_tickets"]
categorical_features = ["contract", "region", "payment"]
label_name = "churn"

pd.DataFrame(
    {
        "rows": [len(train_df), len(val_df), len(test_df)],
        "positive_rate": [train_df[label_name].mean(), val_df[label_name].mean(), test_df[label_name].mean()],
    },
    index=["train", "validation", "test"],
)


## Erst einmal einfacher: Pandas und scikit-learn

Für klassische Tabellendaten ist scikit-learn oft der verständlichste Einstieg. Eine `Pipeline` beschreibt eine feste Verarbeitungskette:

1. numerische Spalten: fehlende Werte ersetzen, dann skalieren,
2. kategorische Spalten: fehlende Werte ersetzen, dann One-Hot-Encoding,
3. Modell trainieren.

Der Vorteil: Die Pipeline merkt sich alle gelernten Werte aus dem Training, also Mittelwerte, Standardabweichungen und Kategorien. In Jupyter/Colab kann man die Pipeline außerdem als Diagramm anzeigen.


In [ ]:
set_config(display="diagram")

# Pipeline für numerische Spalten:
# Erst fehlende Werte ersetzen, dann skalieren.
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
    ]
)

# Pipeline für kategorische Spalten:
# Erst fehlende Kategorien auffüllen, dann One-Hot-Encoding.
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="[MISSING]")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# ColumnTransformer sagt: Wende diese Pipeline auf diese Spalten an.
sklearn_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    verbose_feature_names_out=False,
)

# Gesamte sklearn-Pipeline: Preprocessing und danach ein einfaches Modell.
sklearn_pipeline = Pipeline(
    steps=[
        ("preprocessing", sklearn_preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

# In einem Notebook wird diese letzte Zeile als visuelles Pipeline-Diagramm dargestellt.
sklearn_pipeline


## scikit-learn-Pipeline ausführen und ansehen

Die scikit-learn-Pipeline ist hier bewusst als Vergleich eingebaut. Sie ist für viele Tabular-Projekte eine sehr gute Baseline. Danach bauen wir dieselbe Idee in Keras nach, damit Preprocessing und neuronales Netz gemeinsam gespeichert werden können.


In [ ]:
X_train_pd = train_df[numeric_features + categorical_features]
y_train_pd = train_df[label_name]
X_val_pd = val_df[numeric_features + categorical_features]
y_val_pd = val_df[label_name]

sklearn_pipeline.fit(X_train_pd, y_train_pd)

validation_accuracy = sklearn_pipeline.score(X_val_pd, y_val_pd)
print("Validation Accuracy:", round(validation_accuracy, 3))

feature_names = sklearn_pipeline.named_steps["preprocessing"].get_feature_names_out()
transformed_preview = sklearn_pipeline.named_steps["preprocessing"].transform(X_train_pd.head(5))

pd.DataFrame(transformed_preview, columns=feature_names).head()


## Kurz erklärt: Was ist ein `tf.data.Dataset`?

`tf.data.Dataset` ist eine TensorFlow-Klasse für Datenpipelines. Ein Dataset-Objekt kann einzelne Beispiele, Dateipfade oder bereits gebatchte Daten enthalten.

Darum darf man Methoden mit Punktnotation aufrufen:

- `ds.shuffle(...)`: mischt Elemente des Dataset-Objekts.
- `ds.batch(64)`: fasst 64 einzelne Elemente zu einem Batch zusammen.
- `ds.map(funktion)`: wendet eine Funktion auf jedes Element an.
- `ds.prefetch(...)`: bereitet Daten vor, während das Modell schon rechnet.

Das sind keine frei erfundenen Funktionen aus diesem Notebook, sondern Methoden der TensorFlow-Klasse `tf.data.Dataset`.

Wichtig für den Split:

- `train_ds` wird geshuffelt, weil Training von gemischten Batches profitiert.
- `val_ds` und `test_ds` werden nicht geshuffelt, weil Evaluation stabil und reproduzierbar bleiben soll.


## TensorFlow-Datasets aus DataFrames

Ab hier bauen wir die TensorFlow-Variante. Warum nicht einfach nur scikit-learn?

- scikit-learn ist für klassische Tabular-Pipelines sehr gut lesbar.
- Keras/TensorFlow ist nützlich, wenn die Vorverarbeitung zusammen mit einem neuronalen Netz gespeichert und später direkt mit TensorFlow geladen werden soll.
- TFRecords gehören in die TensorFlow-Welt und werden über `tf.data` gelesen.

Keras kann Dictionaries als Input verarbeiten. Jeder Feature-Name wird ein eigener `Input`. Dadurch bleibt klar, welches Rohmerkmal wohin geht.


In [ ]:
def dataframe_to_dataset(dataframe, shuffle=True, batch_size=64):
    # Wir kopieren, damit die Original-DataFrames nicht verändert werden.
    dataframe = dataframe.copy()

    # Das Label wird vom Feature-Teil getrennt.
    labels = dataframe.pop(label_name).astype("float32")

    features = {}

    # Numerische Features bleiben float32. reshape(-1, 1) macht aus einer Spalte
    # explizit eine zweidimensionale Eingabe: (samples, 1).
    for col in numeric_features:
        features[col] = dataframe[col].astype("float32").to_numpy().reshape(-1, 1)

    # Kategorische Features werden Strings. Fehlende Werte bekommen eine sichtbare Kategorie.
    for col in categorical_features:
        features[col] = dataframe[col].fillna("[MISSING]").astype(str).to_numpy().reshape(-1, 1)

    # from_tensor_slices erzeugt ein Dataset aus einzelnen Samples.
    # Ein Sample sieht aus wie: ({"age": ..., "contract": ...}, label).
    ds = tf.data.Dataset.from_tensor_slices((features, labels.to_numpy()))
    if shuffle:
        # Nur train_ds bekommt shuffle=True.
        # val_ds/test_ds bekommen shuffle=False, damit die Bewertung stabil bleibt.
        ds = ds.shuffle(buffer_size=len(dataframe), seed=RANDOM_STATE)

    # .batch(...) und .prefetch(...) sind Methoden des Dataset-Objekts.
    # Rückgabe ist wieder ein Dataset, jetzt aber mit Batches statt Einzelsamples.
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


train_ds = dataframe_to_dataset(train_df)
val_ds = dataframe_to_dataset(val_df, shuffle=False)
test_ds = dataframe_to_dataset(test_df, shuffle=False)

# Ein Kontroll-Batch: Wir holen ein Element aus train_ds und schauen nur auf die Feature-Namen.
# train_batch_features ist ein Dictionary: {Feature-Name -> Tensor mit Batch-Werten}.
train_batch_features, train_batch_labels = next(iter(train_ds))
print("Feature-Namen:", list(train_batch_features.keys()))
print("age-Batch-Shape:", train_batch_features["age"].shape)
print("Label-Batch-Shape:", train_batch_labels.shape)


## Keras-Preprocessing-Modell

Jetzt bauen wir dieselbe Idee wie in scikit-learn direkt in Keras nach.

Numerische Features:

- fehlende Werte mit Trainingsmittelwert imputieren,
- anschließend normalisieren.

Kategorische Features:

- fehlende Werte als eigene Kategorie `[MISSING]`,
- `StringLookup` lernt das Vokabular,
- `StringLookup(output_mode="one_hot")` erzeugt One-Hot-ähnliche Vektoren.

Diese Vorverarbeitung wird Teil des Keras-Modells. Dadurch ist beim Speichern und Laden klar, wie Rohdaten transformiert werden. Das ist der Hauptvorteil gegenüber einer losen Vorverarbeitung im Notebook.

Hinweis zur Registrierung eigener Keras-Bausteine: Wir verwenden `keras.utils.register_keras_serializable(...)`, weil dieses Notebook `tf.keras` nutzt. Dadurch kann das Modell später auch die eigene `NumericImputer`-Schicht wieder laden.


In [ ]:
inputs = {}
encoded_features = []


# Keras hat eine Normalization-Schicht, aber keine einfache eingebaute Mean-Imputation.
# Deshalb definieren wir eine kleine eigene Schicht: NaN wird durch den Trainingsmittelwert ersetzt.
# Die Registrierung ist nur für späteres Speichern/Laden wichtig.
# Bei tf.keras verwenden wir keras.utils, nicht keras.saving.
@keras.utils.register_keras_serializable(package="kurs")
class NumericImputer(layers.Layer):
    def __init__(self, fill_value, **kwargs):
        super().__init__(**kwargs)
        self.fill_value = float(fill_value)

    def call(self, inputs):
        fill = tf.fill(tf.shape(inputs), tf.cast(self.fill_value, inputs.dtype))
        return tf.where(tf.math.is_nan(inputs), fill, inputs)

    def get_config(self):
        config = super().get_config()
        config.update({"fill_value": self.fill_value})
        return config


for feature in numeric_features:
    # Ein eigener Input pro Rohmerkmal. Dadurch kann das Modell später Rohdaten-Dictionaries annehmen.
    inputs[feature] = keras.Input(shape=(1,), name=feature, dtype=tf.float32)

    # Der Mittelwert wird nur aus Trainingsdaten berechnet.
    mean_value = np.float32(train_df[feature].mean())
    imputed = NumericImputer(mean_value, name=f"{feature}_imputation")(inputs[feature])

    # Normalization lernt Mittelwert und Standardabweichung aus Trainingsdaten.
    normalizer = layers.Normalization(name=f"{feature}_normalization")
    normalizer.adapt(train_df[feature].dropna().astype("float32").to_numpy().reshape(-1, 1))
    encoded_features.append(normalizer(imputed))

for feature in categorical_features:
    inputs[feature] = keras.Input(shape=(1,), name=feature, dtype=tf.string)

    # StringLookup lernt ein Vokabular aus den Trainingsdaten.
    # output_mode="one_hot" erzeugt direkt einen numerischen Vektor.
    lookup = layers.StringLookup(output_mode="one_hot", name=f"{feature}_one_hot")
    lookup.adapt(train_df[feature].fillna("[MISSING]").astype(str).to_numpy())

    encoded = lookup(inputs[feature])
    # Flatten macht aus (batch, 1, kategorien) die Form (batch, kategorien).
    encoded = layers.Flatten(name=f"{feature}_encoded")(encoded)
    encoded_features.append(encoded)

# Jetzt werden alle vorbereiteten Merkmale zu einem großen Feature-Vektor verbunden.
all_features = layers.Concatenate(name="all_preprocessed_features")(encoded_features)

# Ab hier beginnt das eigentliche neuronale Netz.
x = layers.Dense(32, activation="relu")(all_features)
x = layers.Dropout(0.15)(x)
x = layers.Dense(16, activation="relu")(x)
output = layers.Dense(1, activation="sigmoid", name="churn_probability")(x)

model = keras.Model(inputs=inputs, outputs=output, name="tabular_pipeline_model")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.AUC(name="auc"),
    ],
)

model.summary()


In [ ]:
# train_ds kommt aus dataframe_to_dataset(train_df).
# val_ds kommt aus dataframe_to_dataset(val_df, shuffle=False).
# model.fit sieht also keine pandas-DataFrames mehr, sondern Batches aus Feature-Dictionaries.
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    verbose=0,
)

pd.DataFrame(history.history).tail()


In [ ]:
# test_ds wurde genau wie train_ds/val_ds gebaut, aber erst jetzt benutzt.
# return_dict=True macht direkt sichtbar, welche Zahl zu welcher Metrik gehört.
model.evaluate(test_ds, verbose=0, return_dict=True)


## Vollständige Pipeline speichern

Weil Imputation, Normalisierung und Encoding Teil des Keras-Modells sind, kann das Modell Rohdaten im gleichen Format wie das Training annehmen. Das reduziert Fehler beim späteren Laden deutlich.


In [ ]:
pipeline_path = Path(tempfile.mkdtemp(prefix="tabular_pipeline_model_")) / "tabular_pipeline.keras"
model.save(pipeline_path)

loaded_pipeline = keras.models.load_model(pipeline_path)
loaded_result = loaded_pipeline.evaluate(test_ds, verbose=0)
dict(zip(loaded_pipeline.metrics_names, loaded_result))


## TFRecords schreiben

Ein TFRecord ist eine Datei, in der viele Beispiele hintereinander binär gespeichert werden. Man kann es sich grob so vorstellen:

```text
pandas-Zeile
  -> tf.train.Example mit benannten Features
  -> serialisierte Bytes
  -> train.tfrecord
```

Warum macht man das?

- TensorFlow kann TFRecords effizient sequenziell lesen.
- Große Datensätze können in mehrere Dateien, sogenannte Shards, aufgeteilt werden.
- Die Trainingspipeline kann lesen, parsen, mischen und batchen, ohne alles zuerst als pandas DataFrame im RAM zu halten.

Wichtig: TFRecords speichern nicht automatisch ein schönes Tabellenschema wie pandas. Wir müssen beim Schreiben selbst festlegen, welche Features es gibt und ob sie als `float` oder `bytes` gespeichert werden. Beim Lesen müssen wir exakt dasselbe Schema wieder angeben.


In [ ]:
# Ziel dieser Zelle:
# 1. Jede pandas-Zeile wird in ein tf.train.Example übersetzt.
# 2. Jedes Example wird als Bytes in eine .tfrecord-Datei geschrieben.
# 3. Die Datei liegt in einem lokalen, temporären Demo-Ordner.

def bytes_feature(value):
    # Strings werden in TFRecords als bytes gespeichert.
    # Beispiel: "monthly" -> b"monthly".
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(value).encode("utf-8")]))


def float_feature(value):
    # Zahlen werden als float gespeichert.
    # Fehlende numerische Werte bleiben NaN. Die Imputation passiert später im Keras-Modell.
    if pd.isna(value):
        value = np.nan
    return tf.train.Feature(float_list=tf.train.FloatList(value=[float(value)]))


def serialize_row(row):
    # Dieses Dictionary beschreibt ein einzelnes Trainingsbeispiel.
    # Keys sind Feature-Namen, Values sind TensorFlow-Feature-Objekte.
    feature = {}

    # Numerische Spalten: als float speichern.
    for col in numeric_features:
        feature[col] = float_feature(row[col])

    # Kategorische Spalten: als bytes speichern.
    for col in categorical_features:
        value = "[MISSING]" if pd.isna(row[col]) else row[col]
        feature[col] = bytes_feature(value)

    # Das Label wird im TFRecord mitgespeichert, beim Lesen aber wieder abgetrennt.
    feature[label_name] = float_feature(row[label_name])

    # tf.train.Example ist das Standard-Containerformat für ein einzelnes Beispiel.
    example = tf.train.Example(features=tf.train.Features(feature=feature))

    # TFRecordWriter erwartet Bytes, deshalb serialisieren wir das Example.
    return example.SerializeToString()


# Für die Demo speichern wir den TFRecord lokal im Notebook-Ordner.
# Der Ordner wird über .gitignore ausgeschlossen.
tfrecord_dir = Path("notebooks/Day_3_5/generated_data/17_tfrecord_pipeline")
if not tfrecord_dir.exists() and not Path("notebooks/Day_3_5").exists():
    tfrecord_dir = Path("generated_data/17_tfrecord_pipeline")
tfrecord_dir.mkdir(parents=True, exist_ok=True)
train_record_path = tfrecord_dir / "train.tfrecord"

with tf.io.TFRecordWriter(str(train_record_path)) as writer:
    for _, row in train_df.iterrows():
        writer.write(serialize_row(row))

print(train_record_path)
print("Bytes:", train_record_path.stat().st_size)


## TFRecords lesen

Beim Lesen passiert der Rückweg:

```text
train.tfrecord
  -> einzelne Byte-Strings
  -> tf.io.parse_single_example(...)
  -> Feature-Dictionary plus Label
  -> tf.data.Dataset für model.fit(...)
```

Der wichtigste Teil ist `feature_description`: Dort steht, welche Namen und Datentypen TensorFlow aus jedem Example lesen soll. Wenn beim Schreiben `monthly_spend` als float gespeichert wurde, muss es beim Lesen auch als `tf.float32` beschrieben werden.


In [ ]:
# Das Leseschema muss exakt zu den geschriebenen TFRecord-Features passen.
# FixedLenFeature([], tf.float32) bedeutet: pro Beispiel genau eine float-Zahl.
# FixedLenFeature([], tf.string) bedeutet: pro Beispiel genau ein String/bytes-Wert.
feature_description = {
    **{col: tf.io.FixedLenFeature([], tf.float32) for col in numeric_features},
    **{col: tf.io.FixedLenFeature([], tf.string) for col in categorical_features},
    label_name: tf.io.FixedLenFeature([], tf.float32),
}


def parse_example(serialized):
    # serialized ist ein einzelner Byte-String aus der TFRecord-Datei.
    # parse_single_example macht daraus wieder ein Dictionary mit Tensoren.
    parsed = tf.io.parse_single_example(serialized, feature_description)

    # Das Label soll nicht als Input ins Modell gehen, sondern separat an model.fit.
    label = parsed.pop(label_name)

    # Wichtig: parse_single_example liefert hier skalare Tensoren mit Shape ().
    # Unsere Keras-Inputs wurden aber als shape=(1,) definiert.
    # Deshalb machen wir aus jedem skalaren Feature explizit einen 1D-Tensor mit Shape (1,).
    # Nach dem Batching wird daraus (batch, 1), genau wie beim DataFrame-Dataset.
    for col in numeric_features + categorical_features:
        parsed[col] = tf.expand_dims(parsed[col], axis=-1)
        parsed[col].set_shape((1,))

    return parsed, label


tfrecord_train_ds = (
    # TFRecordDataset streamt Beispiele aus der Datei.
    tf.data.TFRecordDataset(str(train_record_path))
    # Jedes binäre Example wird in Features und Label übersetzt.
    .map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    # Danach sieht das Dataset für Keras fast genauso aus wie train_ds aus dem DataFrame.
    .shuffle(2000, seed=RANDOM_STATE)
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)

features_batch, labels_batch = next(iter(tfrecord_train_ds))
print({key: value.shape for key, value in features_batch.items()})
print(labels_batch.shape)


In [ ]:
# Ein kurzer zusätzlicher Fit auf dem TFRecord-Dataset zeigt:
# Das Modell kann direkt mit dem geparsten Dataset trainieren.
# Die Feature-Formen sind jetzt identisch zum DataFrame-Dataset: (batch, 1) pro Spalte.
history_tfrecord = model.fit(
    tfrecord_train_ds,
    validation_data=val_ds,
    epochs=3,
    verbose=0,
)

pd.DataFrame(history_tfrecord.history)


## Limitierungen

- TFRecords sind für kleine Tabellen nicht nötig. CSV oder Parquet sind oft einfacher zu debuggen.
- TFRecords sind binär. Man kann sie nicht sinnvoll mit einem Texteditor öffnen. Deshalb sind Schema und Parser-Code besonders wichtig.
- Das TFRecord-Schema muss stabil gehalten werden. Geänderte Feature-Namen oder Datentypen brechen den Parser.
- `adapt(...)` darf nur auf Trainingsdaten laufen. Sonst entsteht Data Leakage.
- Imputation mit Mittelwert ist nur eine einfache Baseline. Bei stark verzerrten Verteilungen oder fachlichen Sonderwerten braucht man bessere Strategien.
- One-Hot-Encoding kann bei sehr vielen Kategorien groß werden. Dann sind Embeddings oder Hashing oft sinnvoller.
- Die Beispielpipeline behandelt keine Zeitabhängigkeit. Bei Zeitreihen muss der Split zeitlich sauber sein.
